# SCP Step 2 (Passive Properties) - Colab Classroom Notebook

This notebook is designed for first-time users and class labs.  
It builds a tuned single-cell model, runs passive current-injection tests, and reports passive metrics (for example input resistance and time constants).

## Learning goals
1. Understand how SCP loads a tuned model from `cell_config.json`.
2. Run reproducible passive stimulation protocols.
3. Compare model responses to target passive behavior.

## How to run this notebook
1. Run cells from top to bottom.
2. Edit only the marked configuration cells first.
3. If anything fails, use the troubleshooting section at the end.

## Notebook-only usage (Colab)
- This notebook can auto-clone SCP and ACT if needed.
- For private repos, set a token in env vars (`SCP_GIT_TOKEN` or `GITHUB_TOKEN`).


In [ ]:
import os, sys, json
import subprocess
import importlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def _is_scp_root(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / 'cells').is_dir()
        and (path / 'modules').is_dir()
        and (path / 'run_pipeline.py').is_file()
    )


def _find_scp_root(start: Path):
    start = start.resolve()
    for cand in [start] + list(start.parents):
        if _is_scp_root(cand):
            return cand

    for base in (start, start.parent):
        try:
            for child in base.iterdir():
                if child.is_dir() and _is_scp_root(child):
                    return child.resolve()
        except Exception:
            pass
    return None


def _repo_url_with_token(repo_url: str) -> str:
    token = (
        os.environ.get('SCP_GIT_TOKEN')
        or os.environ.get('SCP_GITHUB_TOKEN')
        or os.environ.get('GITHUB_TOKEN')
    )
    if token and repo_url.startswith('https://') and '@' not in repo_url:
        return repo_url.replace('https://', f'https://{token}@', 1)
    return repo_url


def _redact_repo_url(url: str) -> str:
    if '://' not in url or '@' not in url:
        return url
    scheme, rest = url.split('://', 1)
    host_part = rest.split('@', 1)[1]
    return f'{scheme}://***@{host_part}'


def _git_clone(repo_url: str, repo_dir: Path, branch=None, repo_label: str = 'repo') -> Path:
    repo_dir = repo_dir.expanduser().resolve()
    if repo_dir.exists() and any(repo_dir.iterdir()):
        return repo_dir

    repo_dir.parent.mkdir(parents=True, exist_ok=True)
    auth_url = _repo_url_with_token(repo_url)
    cmd = ['git', 'clone', '--depth', '1']
    if branch:
        cmd += ['--branch', branch]
    cmd += [auth_url, str(repo_dir)]

    shown = [*cmd[:-2], _redact_repo_url(auth_url), str(repo_dir)]
    print(f'Cloning {repo_label}:', ' '.join(shown))
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            f'Failed to clone {repo_label}. For private repos, set SCP_GIT_TOKEN or GITHUB_TOKEN.'
        ) from exc
    return repo_dir


def _ensure_python_pkg(import_name: str, pip_name=None) -> None:
    try:
        importlib.import_module(import_name)
        return
    except Exception:
        pass
    pkg = pip_name or import_name
    print(f'Installing missing package: {pkg}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])


IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
AUTO_CLONE_REPO = os.environ.get('SCP_AUTO_CLONE', '1') not in ('0', 'false', 'False')
DEFAULT_REPO_URL = 'https://github.com/HunterBushnell/SCP.git'
REPO_URL = os.environ.get('SCP_REPO_URL', DEFAULT_REPO_URL)
REPO_BRANCH = os.environ.get('SCP_REPO_BRANCH', '') or None
DEFAULT_REPO_DIR = Path('/content/SCP') if IN_COLAB else (Path.cwd() / 'SCP')
REPO_DIR = Path(os.environ.get('SCP_REPO_DIR', str(DEFAULT_REPO_DIR)))

repo_root = _find_scp_root(Path.cwd())
if repo_root is None and AUTO_CLONE_REPO:
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()) and not _is_scp_root(REPO_DIR):
        raise FileExistsError(
            f'SCP_REPO_DIR exists but is not an SCP checkout: {REPO_DIR}'
        )
    _git_clone(REPO_URL, REPO_DIR, branch=REPO_BRANCH, repo_label='SCP')
    repo_root = _find_scp_root(REPO_DIR)

if repo_root is None:
    raise FileNotFoundError(
        'Could not locate SCP repo from current directory. '
        'Set SCP_REPO_DIR/SCP_REPO_URL or run notebook from inside SCP.'
    )

repo_root = repo_root.resolve()
os.environ['SCP_ROOT'] = str(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print('SCP root:', repo_root)

from modules.notebook_helpers import ensure_scp_repo_on_syspath, ensure_external_repo_on_syspath
repo_root = ensure_scp_repo_on_syspath(repo_root)
os.environ['SCP_ROOT'] = str(repo_root)

if IN_COLAB:
    _ensure_python_pkg('numpy')
    _ensure_python_pkg('matplotlib')
    _ensure_python_pkg('scipy')
    _ensure_python_pkg('h5py')
    _ensure_python_pkg('neuron')
    _ensure_python_pkg('allensdk')
    subprocess.check_call(['apt-get', 'update'])
    subprocess.check_call(['apt-get', 'install', '-y', 'build-essential'])

# Import SCP simulation helpers
from modules import run_sim

# Resolve or clone ACT modules (read-only external dependency)
ACT_REPO_URL = os.environ.get('SCP_ACT_REPO_URL', 'https://github.com/V-Marco/ACT.git')
ACT_REPO_BRANCH = os.environ.get('SCP_ACT_REPO_BRANCH', '') or None
ACT_REPO_DIR = Path(os.environ.get('SCP_ACT_DIR', str(repo_root.parent / 'mods' / 'ACT')))
AUTO_CLONE_ACT = os.environ.get('SCP_AUTO_CLONE_ACT', '1') not in ('0', 'false', 'False')

try:
    act_path = ensure_external_repo_on_syspath(
        repo_name='ACT',
        marker_rel=Path('act') / 'passive.py',
        env_vars=('SCP_ACT_PATH', 'SCP_ACT_DIR', 'ACT_PATH', 'ACT_ROOT'),
        repo_root=repo_root,
    )
except FileNotFoundError:
    if not AUTO_CLONE_ACT:
        raise
    if ACT_REPO_DIR.exists() and any(ACT_REPO_DIR.iterdir()):
        raise FileExistsError(
            f'SCP_ACT_DIR exists but is not a valid ACT checkout: {ACT_REPO_DIR}'
        )
    _git_clone(ACT_REPO_URL, ACT_REPO_DIR, branch=ACT_REPO_BRANCH, repo_label='ACT')
    os.environ['SCP_ACT_PATH'] = str(ACT_REPO_DIR)
    act_path = ensure_external_repo_on_syspath(
        repo_name='ACT',
        marker_rel=Path('act') / 'passive.py',
        env_vars=('SCP_ACT_PATH', 'SCP_ACT_DIR', 'ACT_PATH', 'ACT_ROOT'),
        repo_root=repo_root,
    )

print('ACT path:', act_path)

from act.passive import ACTPassiveModule


## A. Environment Bootstrap

The next code cell does all setup work:
- finds or clones SCP,
- installs required Python packages in Colab,
- resolves ACT dependency,
- adds repository paths to `sys.path`.

If this cell succeeds, you can usually run the rest of the notebook without manual environment setup.


## B. Choose Cell and Tune (edit next cell)

In the next code cell, students should normally edit:
- `cell_name` (`'PV'` or `'SST'`),
- `model_dir` (usually `'seg_tuned'`).

Everything else can stay at defaults for a standard class exercise.


In [ ]:
# USER SETTINGS (students usually edit only this block)
# - cell_name: choose which tuned cell to test
# - model_dir: choose which tune folder to load

cell_name = 'PV' #SST, PV, PN

model_dir = 'seg_tuned'

tune_dir = (repo_root / 'cells' / cell_name / 'tunes' / model_dir).resolve()
if not tune_dir.is_dir():
    raise FileNotFoundError(f'Expected tune dir not found: {tune_dir}')
os.chdir(tune_dir)
print('CWD:', Path.cwd())



In [ ]:
cfg_path = Path('cell_configs') / 'sim_config.json'
if not cfg_path.is_file():
    cfg_path = Path('sim_config.json')
if cfg_path.is_file():
    sim_cfg_preview = json.loads(cfg_path.read_text())



## C. Validate Required Model Files

This checks that the selected tune folder already contains:
- `manifest.json`
- `modfiles/`

If missing, run Step 0 preparation/download first.


In [ ]:
bundle_dir = tune_dir
required_paths = [
    bundle_dir / 'manifest.json',
    bundle_dir / 'modfiles',
]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    missing_text = "\n".join(missing)
    raise FileNotFoundError(
        f"Missing required local cell bundle files:\n{missing_text}\n"
        'Prepare/download the bundle in Step 0, then rerun this notebook.'
    )

print(f'Using existing local model bundle: {bundle_dir.resolve()}')


## D. Compile and Load NEURON Mechanisms

This compiles `modfiles/` with `nrnivmodl` only when needed, then loads `libnrnmech.so`.

Expected result: a printed line confirming mechanism library load.


In [ ]:
import subprocess
from neuron import h

os.chdir(tune_dir)
mod_dir = Path('modfiles')
if not mod_dir.is_dir():
    raise FileNotFoundError(f'Missing modfiles directory: {mod_dir.resolve()}')

dll_candidates = [
    mod_dir / 'x86_64' / '.libs' / 'libnrnmech.so',
    mod_dir / 'x86_64' / 'libnrnmech.so',
]
dll_path = next((p for p in dll_candidates if p.is_file()), None)

if dll_path is None:
    print('Compiling modfiles with nrnivmodl...')
    subprocess.check_call(['nrnivmodl'], cwd=str(mod_dir))
    dll_path = next((p for p in dll_candidates if p.is_file()), None)
    if dll_path is None:
        raise FileNotFoundError('nrnivmodl finished but libnrnmech.so was not found.')

h.load_file('stdrun.hoc')  # Required to use h.run()
h.nrn_load_dll(str(dll_path))
print(f'Loaded mechanisms from {dll_path}')


## E. Build the Cell Object

This stage creates a NEURON cell using `modules.load_cell`,
using shared SCP notebook helpers.

Expected result: a printout of soma area/diameter/length.


In [ ]:
from modules.notebook_helpers import (
    build_cell_for_notebook,
    resolve_cell_config_for_notebook,
)

cell_config_for_build = resolve_cell_config_for_notebook(cell_name)
cell = build_cell_for_notebook(cell_config_for_build)

sect = cell.soma[0]
seg = 0.5
print(f"{sect}: A={round(sect(seg).area(),2)} | D={round(sect.diam)} | L={round(sect.L)}")


## F. Passive Targets and Interpretation

This section computes passive-property summary values from the built cell and target values.
Use this as a quick check before running the stimulation protocol.


In [ ]:
# Soma area from the previous cell
computed_soma_area = cell.soma[0](0.5).area() * 1e-8 #(cm2)
# User-provided desired properties
user_provided_Rin = 195.4 * 10e6    # (to Ohm from MOhm)
user_provided_tau = 15.6 * 1e-3     # (to s from ms)
user_provided_Vrest = -65.75        # (mV)
spps = ACTPassiveModule.compute_spp(user_provided_Rin, computed_soma_area, user_provided_tau, user_provided_Vrest)
print(spps)

## G. Run Passive Protocol (main output)

The next code cell runs hyperpolarizing current steps and plots membrane traces.

Student edits to try:
- `sim_amps` (pA amplitudes),
- `stim_delay` / `stim_dur`,
- plot limits.

Interpretation:
- Larger voltage deflection at the same current usually means higher input resistance.
- Slower return toward baseline usually means larger membrane time constant.


In [ ]:
# STUDENT EDITS: adjust stimulus and amplitudes for experiments
# Units: pA in sim_amps, ms in timing fields
# Simulation parameters
sim_params = {
    'stim_amp': -0.1,
    'stim_delay': 300,
    'stim_dur': 1000,
    'h_tstop': 1500,
    'h_dt': 0.025
    }
# Currents injected for each sim
sim_amps = [-50,-100]

cell = build_cell_for_notebook(cell_config_for_build)
looped_records = run_sim.looped_current_injection(cell,sim_params,sim_amps)
print(looped_records['I'][-50].keys())

# Analyze and plot each run
for amp in sim_amps:
    print(f"{amp} pA spike frequency = {looped_records['F'][amp]:.2f} Hz")
    if amp <0:
        # print('Desired Properties:       R_in=98.9, tau1=5.9, tau2=??.?, sag_ratio=0.960, V_rest=-71.25')
        print(f"{ACTPassiveModule.compute_gpp(looped_records['V'][amp], h.dt, sim_params['stim_delay'], sim_params['stim_delay']+sim_params['stim_dur']-10, amp/1000)}\n")
    plt.plot(looped_records['T'][amp], looped_records['V'][amp], label=f"{amp} pA")
    
# Plot parameters
plt.xlabel("Time (ms)"), plt.ylabel("Vm (mV)"), plt.title(f"Vm Traces ({cell_name})")
# plt.xlim(250,800)
# plt.ylim(-80,40)
plt.legend(),plt.grid(),plt.show()



## Troubleshooting (Step 2)

If you get an error:
1. `manifest.json` or `modfiles` missing: run Step 0 prep/download first.
2. `libnrnmech.so` missing: rerun the compile/load cell and check `nrnivmodl` output.
3. ACT import failure: check ACT path variables (`SCP_ACT_PATH`, `ACT_PATH`) or allow ACT auto-clone.
4. Unexpected plots/spiking: confirm `cell_name` and tune folder match the intended dataset.

For classroom demos, keep one known-good configuration first, then let students vary one parameter at a time.
